In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

# Daten vorbereiten (wie zuvor)
data = {
    "Datensatzgröße": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
    "Excel Import": [0.172, 0.195, 0.215, 0.203, 0.211, 0.262, 0.227, 0.273, 0.270, 0.297, 0.594, 1.172, 1.797, 2.668, 3.801, 5.039, 6.500, 8.133, 10.027, 39.809, 91.641, 168.117, 280.000, 380.000, 500.000, 650.000, 820.000, 1020.000],
    "Power Query": [0.199, 0.164, 0.180, 0.199, 0.199, 0.203, 0.215, 0.215, 0.246, 0.230, 0.332, 0.535, 0.785, 1.098, 1.402, 1.852, 2.344, 2.902, 3.488, 13.332, 29.410, 51.695, 80.957, 116.941, 158.941, 207.082, 264.715, 327.465],
    "Schleifenoptimierung 2": [0.186, 0.174, 0.191, 0.180, 0.203, 0.176, 0.186, 0.197, 0.225, 0.223, 0.311, 0.428, 0.613, 0.898, 1.176, 1.564, 1.949, 2.377, 2.918, 10.943, 25.244, 43.598, 66.960, 96.668, 130.824, 170.186, 216.646, 264.131],
    "Binary Read 3": [0.016, 0.025, 0.018, 0.018, 0.018, 0.029, 0.016, 0.037, 0.047, 0.051, 0.109, 0.191, 0.328, 0.480, 0.686, 0.924, 1.188, 1.512, 1.848, 7.162, 16.393, 28.492, 44.314, 64.752, 87.082, 113.393, 143.215, 175.734],
    "Powershell Max Parallel 10": [0.2242, 0.6879, 0.4457, 0.5426, 0.4836, 0.4852, 0.4832, 0.5207, 0.5578, 0.4824, 0.7184, 0.5496, 0.6035, 0.6430, 0.6230, 0.5508, 0.5395, 0.5980, 0.5719, 0.6746, 0.6859, 0.5418, 0.6723, 0.5875, 0.5961, 0.5504, 0.5566, 0.5906]
}

df = pd.DataFrame(data)

# Zielbereich für die Extrapolation definieren (z.B. bis 20.000)
x_original = df["Datensatzgröße"]
x_extrapolate = np.logspace(np.log10(x_original.min()), np.log10(20000), 500) # 500 Punkte für eine glatte Kurve

# Grafik erstellen
plt.figure(figsize=(14, 8))

columns_to_plot = ["Excel Import", "Power Query", "Schleifenoptimierung 2", "Binary Read 3", "Powershell Max Parallel 10"]

# Für jede Spalte interpolieren und extrapolieren
for col in columns_to_plot:
    y_original = df[col]
    
    # Kubische Spline-Interpolation für den Originalbereich
    f_interp = interp1d(x_original, y_original, kind='cubic')
    y_interp = f_interp(np.logspace(np.log10(x_original.min()), np.log10(x_original.max()), 200)) # Glatte Kurve im Originalbereich
    
    # Lineare Trendfortsetzung (basiert auf den letzten beiden Datenpunkten im Log-Space)
    x_log = np.log10(x_original.iloc[-2:])
    y_log = np.log10(y_original.iloc[-2:])
    slope = (y_log.iloc[-1] - y_log.iloc[0]) / (x_log.iloc[-1] - x_log.iloc[0])
    intercept = y_log.iloc[-1] - slope * x_log.iloc[-1]
    
    x_extrapolate_log = np.log10(x_extrapolate[x_extrapolate >= x_original.max()])
    y_extrapolate_log = slope * x_extrapolate_log + intercept
    y_extrapolate = 10**y_extrapolate_log

    # Originalbereich plotten (durchgezogene Linie, dicker)
    line, = plt.plot(np.logspace(np.log10(x_original.min()), np.log10(x_original.max()), 200), y_interp, linewidth=2, label=col)
    
    # Extrapolierten Bereich plotten (gestrichelte Linie, dünner, gleiche Farbe)
    plt.plot(x_extrapolate[x_extrapolate >= x_original.max()], y_extrapolate, color=line.get_color(), linestyle='--', linewidth=1)

    # Linienbeschriftung am Ende hinzufügen
    plt.text(x_extrapolate[-1], y_extrapolate[-1], f' {col}', verticalalignment='center', color=line.get_color(), fontweight='bold')

# Achsenskalierung und Beschriftung
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Datensatzgröße (Anzahl)", fontsize=12)
plt.ylabel("Zeit (Sekunden)", fontsize=12)
plt.title("Performance-Vergleich (glatt, interpoliert, extrapoliert)", fontsize=14)

# Legende und Gitter hinzufügen
plt.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98)) # Legende verschieben
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.xlim(x_original.min(), 22000) # X-Achse erweitern
plt.ylim(0.01, 3000) # Y-Achse erweitern für extrapolierte Werte

plt.tight_layout()
plt.show()